In [1]:
"""SGD with a sharpness-aware learning rate (SALR), after Yue, Nouiehed and
Al Kontar (arXiv 2011.05348), on the hybrid model in ``recommender.model``.

How it works
------------
SALR is plain mini-batch SGD whose learning rate follows the local sharpness of
the loss: large steps in sharp regions (to leave them), small steps in flat
regions (to settle there).

Every ``probe_every`` updates, on the current mini-batch B and iterate theta:

1. Probe downhill: ``n_descent`` steps of length ``probe_step`` along -g/|g|.
2. Probe uphill:   ``n_ascent`` steps of length ``probe_step`` along +g/|g|.
   Both probes start from theta, and theta itself is not moved by them.
3. Sharpness  S = F_B(uphill end) - F_B(downhill end).
4. Learning rate  lr = base_lr * S / median(all valid S so far).

Then the usual update  theta <- theta - lr * grad F_B(theta).  Between probes
the last learning rate is kept. An estimate that is not finite or not positive
is invalid: it is logged, left out of the median, and the learning rate is kept.

F_B is the regularized objective of ``model.loss_and_grad``. Probe steps are
normalized by the norm of the mini-batch gradient (the paper's Definition 1).

Gradient work
-------------
The first step of each probe reuses the gradient already computed for the
update, so a probe adds ``(n_descent - 1) + (n_ascent - 1)`` gradients of |B|
units each. The two loss evaluations at the probe ends add no gradient units,
but their time is part of the training time.
"""

from __future__ import annotations

import time
from dataclasses import asdict, dataclass

import numpy as np

from recommender.model import Params, data_loss, init_params, loss_and_grad, penalty


@dataclass(frozen=True)
class SALRConfig:
    base_lr: float
    probe_step: float  # gamma: length of one probe step in parameter space
    lambda_f: float
    lambda_w: float
    n_descent: int = 1
    n_ascent: int = 1
    probe_every: int = 1  # c: probe once every this many updates
    batch_size: int = 256


def _moved(params: Params, direction: Params, scale: float) -> Params:
    return {name: value + scale * direction[name] for name, value in params.items()}


def _norm(grads: Params) -> float:
    return float(np.sqrt(sum(np.sum(g**2) for g in grads.values())))


def epoch_batches(triplets: np.ndarray, batch_size: int, rng: np.random.Generator):
    """One shuffled pass over the triplets; the final smaller batch is kept."""
    order = rng.permutation(len(triplets))
    for start in range(0, len(triplets), batch_size):
        yield triplets[order[start : start + batch_size]]


def sharpness(params: Params, grads: Params, data, batch: np.ndarray, cfg: SALRConfig):
    """Local sharpness at ``params`` on ``batch``, given the gradient there.
    Returns the estimate and the number of extra gradient evaluations used."""
    ends, extra_gradients = [], 0
    for sign, n_steps in ((-1.0, cfg.n_descent), (1.0, cfg.n_ascent)):
        point, direction = params, grads
        for step in range(n_steps):
            if step > 0:
                _, direction = loss_and_grad(point, data, batch, cfg.lambda_f, cfg.lambda_w)
                extra_gradients += 1
            length = _norm(direction)
            if length == 0.0:
                break
            point = _moved(point, direction, sign * cfg.probe_step / length)
        ends.append(data_loss(point, data, batch) + penalty(point, cfg.lambda_f, cfg.lambda_w))
    return ends[1] - ends[0], extra_gradients


def next_learning_rate(estimate: float, history: list[float], current: float, base_lr: float):
    """Apply lr = base_lr * S / median(S). Returns (learning rate, was it valid);
    a valid estimate is appended to ``history``."""
    if not np.isfinite(estimate) or estimate <= 0.0:
        return current, False
    history.append(estimate)
    return base_lr * estimate / float(np.median(history)), True


def train_salr(
    data,
    seed: int,
    cfg: SALRConfig,
    total_work: int,
    checkpoint_every: int,
    evaluate=None,
) -> dict:
    """Train from the seed's shared starting point for ``total_work``
    triplet-gradient units, checkpointing every ``checkpoint_every`` units.

    ``evaluate(params)`` may return a dict of extra metrics (for example
    ordinary-validation NDCG@10) to store with each checkpoint.

    Returns a log with one entry per update (``work``, ``lr``, ``probed``,
    ``sharpness``, ``valid``) and a list of ``checkpoints``; ``status`` is
    "failed" if a loss or parameter became non-finite.
    """
    params = init_params(data, seed)
    shuffle = np.random.default_rng(seed + 1000)  # separate stream from the initialization
    log = {
        "optimizer": "salr", "seed": seed, "config": asdict(cfg), "status": "ok",
        "work": [], "lr": [], "probed": [], "sharpness": [], "valid": [], "checkpoints": [],
    }
    history: list[float] = []
    lr, work, updates, train_time = cfg.base_lr, 0, 0, 0.0

    def checkpoint():
        entry = {
            "work": work,
            "updates": updates,
            "train_time": train_time,
            "train_loss": data_loss(params, data, data.train_triplets),
            "val_loss": data_loss(params, data, data.validation_triplets),
            "penalty": penalty(params, cfg.lambda_f, cfg.lambda_w),
            "params": {name: value.copy() for name, value in params.items()},
        }
        if evaluate is not None:
            entry.update(evaluate(params))
        log["checkpoints"].append(entry)

    checkpoint()
    next_checkpoint = checkpoint_every
    while work < total_work:
        for batch in epoch_batches(data.train_triplets, cfg.batch_size, shuffle):
            started = time.perf_counter()
            loss, grads = loss_and_grad(params, data, batch, cfg.lambda_f, cfg.lambda_w)
            gradients, probed, estimate, valid = 1, updates % cfg.probe_every == 0, np.nan, False
            if probed:
                estimate, extra = sharpness(params, grads, data, batch, cfg)
                gradients += extra
                lr, valid = next_learning_rate(estimate, history, lr, cfg.base_lr)
            params = _moved(params, grads, -lr)
            train_time += time.perf_counter() - started

            work += gradients * len(batch)
            updates += 1
            for key, value in zip(
                ("work", "lr", "probed", "sharpness", "valid"), (work, lr, probed, estimate, valid)
            ):
                log[key].append(value)

            if not np.isfinite(loss) or not all(np.isfinite(v).all() for v in params.values()):
                log["status"] = "failed"
                return log
            if work >= next_checkpoint:
                checkpoint()
                next_checkpoint += checkpoint_every
            if work >= total_work:
                break
    return log


In [2]:
"""SALR: sharpness estimate, learning-rate rule, work accounting and training."""

import numpy as np
import pytest

from recommender.data import load_dataset
'''from recommender.model import init_params, loss_and_grad
from recommender.salr import SALRConfig, _norm, next_learning_rate, sharpness, train_salr'''

CFG = SALRConfig(base_lr=0.5, probe_step=0.05, lambda_f=1e-3, lambda_w=1e-3)


@pytest.fixture(scope="module")
def ds():
    return load_dataset()


def test_one_step_sharpness_is_first_order(ds):
    """For a tiny probe step, S = F(theta + g*d) - F(theta - g*d) ~ 2 * step * |g|."""
    params, batch = init_params(ds, 616), ds.train_triplets[:256]
    _, grads = loss_and_grad(params, ds, batch, CFG.lambda_f, CFG.lambda_w)
    cfg = SALRConfig(base_lr=0.5, probe_step=1e-4, lambda_f=1e-3, lambda_w=1e-3)
    estimate, extra = sharpness(params, grads, ds, batch, cfg)
    assert estimate == pytest.approx(2 * cfg.probe_step * _norm(grads), rel=1e-3)
    assert extra == 0  # single-step probes reuse the update's gradient


def test_probes_do_not_move_the_iterate(ds):
    params, batch = init_params(ds, 616), ds.train_triplets[:256]
    before = {name: value.copy() for name, value in params.items()}
    _, grads = loss_and_grad(params, ds, batch, CFG.lambda_f, CFG.lambda_w)
    cfg = SALRConfig(0.5, 0.05, 1e-3, 1e-3, n_descent=3, n_ascent=2)
    estimate, extra = sharpness(params, grads, ds, batch, cfg)
    assert estimate > 0 and extra == 3  # (3 - 1) + (2 - 1)
    assert all(np.array_equal(params[name], before[name]) for name in params)


def test_learning_rate_rule():
    history = []
    assert next_learning_rate(2.0, history, 0.1, 0.1) == (0.1, True)  # S / median = 1
    assert next_learning_rate(6.0, history, 0.1, 0.1) == (pytest.approx(0.15), True)  # 6 / 4
    for invalid in (0.0, -1.0, np.nan, np.inf):
        assert next_learning_rate(invalid, history, 0.15, 0.1) == (0.15, False)
    assert history == [2.0, 6.0]  # invalid estimates stay out of the median


def test_work_accounting(ds):
    cfg = SALRConfig(0.5, 0.05, 1e-3, 1e-3, n_descent=3, n_ascent=3, probe_every=2)
    log = train_salr(ds, 616, cfg, total_work=5_000, checkpoint_every=2_000)
    per_update = np.diff([0] + log["work"])
    assert log["probed"][:4] == [True, False, True, False]
    assert list(per_update[:4]) == [5 * 256, 256, 5 * 256, 256]  # 1 + (3-1) + (3-1) gradients
    assert log["lr"][0] == cfg.base_lr and log["lr"][1] == log["lr"][0]  # held between probes
    assert [c["work"] for c in log["checkpoints"]][0] == 0
    assert all(np.diff([c["work"] for c in log["checkpoints"]]) > 0)


def test_training_is_reproducible_and_reduces_loss(ds):
    first = train_salr(ds, 616, CFG, total_work=60_000, checkpoint_every=20_000)
    again = train_salr(ds, 616, CFG, total_work=60_000, checkpoint_every=20_000)
    assert first["status"] == "ok"
    assert first["lr"] == again["lr"]
    assert first["checkpoints"][-1]["train_loss"] == again["checkpoints"][-1]["train_loss"]
    assert first["checkpoints"][-1]["train_loss"] < first["checkpoints"][0]["train_loss"] - 0.01
    assert first["checkpoints"][0]["train_loss"] == pytest.approx(np.log(2), abs=1e-3)


def test_divergence_is_reported_as_failed(ds):
    cfg = SALRConfig(base_lr=1e6, probe_step=0.05, lambda_f=10.0, lambda_w=10.0)
    with np.errstate(all="ignore"):
        log = train_salr(ds, 616, cfg, total_work=200_000, checkpoint_every=50_000)
    assert log["status"] == "failed"
